# Assignment 9: Outlier Treatment & Feature Engineering

This notebook demonstrates:
- Outlier detection and treatment
- Log transformation
- Feature engineering
- Encoding
- Normalization & standardization(Scaling)
- Column review and dropping irrelevant feature
- Dataset versions and next steps


### Load Cleaned Dataset

We start by loading the cleaned dataset created in the previous Assignment. This is the version that had all missing values treated, data types corrected, and duplicates removed.


In [12]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df = pd.read_csv("cleaned_adult.csv")
df.head()


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,state-gov,77516,bachelors,13,never-married,adm-clerical,not-in-family,white,male,2174,0,40,united-states,<=50k
1,50,self-emp-not-inc,83311,bachelors,13,married-civ-spouse,exec-managerial,husband,white,male,0,0,13,united-states,<=50k
2,38,private,215646,middle/high School,9,divorced,handlers-cleaners,not-in-family,white,male,0,0,40,united-states,<=50k
3,53,private,234721,middle/high School,7,married-civ-spouse,handlers-cleaners,husband,black,male,0,0,40,united-states,<=50k
4,28,private,338409,bachelors,13,married-civ-spouse,prof-specialty,wife,black,female,0,0,40,cuba,<=50k


In [13]:
df['race'].value_counts()

race
white                 41736
black                  4683
asian-pac-islander     1518
amer-indian-eskimo      470
other                   406
Name: count, dtype: int64

### To List All Columns with Missing Values:

In [14]:
df.isna().sum()[df.isna().sum() > 0]

Series([], dtype: int64)

### Check NaN

In [15]:
print(df['sex'].unique())
print(df['income'].unique())

['male' 'female']
['<=50k' '>50k']


In [16]:
print(df['race'].value_counts())

race
white                 41736
black                  4683
asian-pac-islander     1518
amer-indian-eskimo      470
other                   406
Name: count, dtype: int64


**Summary**: Dataset is now ready for feature transformation and modeling preparation.


###  Drop `fnlwgt`

The `fnlwgt` column is a sampling weight useful for census but not predictive. Based on analysis in previous assignments, we found it doesn't contribute meaningfully to income prediction.


In [17]:
df = df.drop(columns=['fnlwgt'])

**Summary**: Dropped unnecessary column to reduce noise and improve model clarity.


## Group rare races

In [18]:
# Normalize race values to lowercase and strip spaces
df['race'] = df['race'].str.lower().str.strip()

# Group rare race categories into 'other'
df['race'] = df['race'].replace(
    ['asian-pac-islander', 'amer-indian-eskimo', 'other'], 'other'
)

## Outlier Detection and Treatment

### Methods Used:
1. **IQR Method** (Interquartile Range)
2. **Z-Score Method**
3. **Domain Knowledge**

We detect outliers for numerical features and treat them based on the strategy:
- Cap extreme values (Winsorization)
- Apply log transformation for skewed variables


We cap extreme values in `capital-gain`, `capital-loss`, and `hours-per-week` using the IQR method. This reduces the influence of outliers without removing rows.


We initially considered using **IQR capping** for `capital-gain` and `capital-loss` to handle extreme outliers.  
However, both columns are **highly right-skewed** and contain **mostly 0s**, which caused the IQR to be 0.  
As a result, **IQR capping incorrectly replaced all non-zero values with 0** — destroying the useful information.

**Instead**, we applied **log transformation (`np.log1p()`)**:
- Handles the skew smoothly
- Retains zero values (since `log1p(0) = 0`)
- Compresses large values (e.g., 99999 → ~11.5)

### Result

This improves feature quality while avoiding artificial flattening.  
Log-transformed values now have meaningful variance and are modeling-ready.


### Log Transform Skewed Features

`capital-gain` and `capital-loss` are highly skewed. We apply `log1p` transformation to normalize them while preserving their relationships.


In [19]:
df['capital-gain'] = np.log1p(df['capital-gain'])
df['capital-loss'] = np.log1p(df['capital-loss'])


 **Summary**: Skew reduced. These features are now more suitable for models sensitive to distribution.


### Create New Feature – `capital_net`

This is a meaningful engineered feature derived from:  
**capital_net = capital-gain - capital-loss**  
It captures net economic benefit and may improve model insight.


In [20]:
df['capital_net'] = df['capital-gain'] - df['capital-loss']

**Summary**: `capital_net` feature added to give model an intuitive signal on net gain.


In [21]:
# Check updated stats
df[['capital-gain', 'capital-loss', 'capital_net']].describe()

,capital-gain,capital-loss,capital_net
count,48813.000000,48813.000000,48813.000000
mean,0.728977,0.351181,0.377796
std,2.446142,1.586680,3.002196
min,0.000000,0.000000,-8.379539
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,11.512925,8.379539,11.512925


###  Encode Categorical Features

- **Label Encoding** for binary features (`sex`, `income`)
- **One-Hot Encoding** for nominal features (`education`, `occupation`, etc.)

This converts strings into numeric format models can understand.


### Label Encoding for Binary Columns

In [22]:
df['sex'] = df['sex'].str.lower().str.strip()
df['sex'] = df['sex'].map({'male': 0, 'female': 1})

df['income'] = df['income'].str.lower().str.strip().str.replace('.', '', regex=False)
df['income'] = df['income'].map({'<=50k': 0, '>50k': 1})

### One-Hot Encoding for Categorical Columns

In [23]:
df = pd.get_dummies(df, columns=['workclass', 'education', 'marital-status',
                                 'occupation', 'relationship', 'race', 'native-country'],
drop_first=True)

 **Summary**: Categorical columns are now numeric and machine-friendly.


###  Normalize & Standardize Features

- **StandardScaler** is used for algorithms assuming normal distribution.
- **MinMaxScaler** is used for algorithms sensitive to scale differences.

We demonstrate both for key features.


## Normalization vs Standardization

**Normalization** scales data between 0 and 1 (MinMax Scaling)  
**Standardization** centers data to mean 0 and std dev 1 (Z-score Scaling)

| Method | Use When |
|--------|----------|
| MinMaxScaler | Data doesn't follow Gaussian distribution |
| StandardScaler | Data is Gaussian or used in algorithms assuming normality (e.g., Logistic Regression) |

We apply both below for demonstration.


In [24]:
scaler = StandardScaler()
df_std = df.copy()
df_std[['age', 'education-num', 'hours-per-week']] = scaler.fit_transform(df_std[['age', 'education-num', 'hours-per-week']])


In [25]:
minmax = MinMaxScaler()
df_norm = df.copy()
df_norm[['age', 'education-num', 'hours-per-week']] = minmax.fit_transform(df_norm[['age', 'education-num', 'hours-per-week']])

 **Summary**: Two standardized versions of the dataset are now available: `df_std` and `df_norm`


## Column-Wise Summary of What We Did

| Column | What We Did | Reason |
|--------|-------------|--------|
| `age` | Standardized + MinMax | Continuous feature — used in modeling |
| `workclass` | Replaced '?' → Imputed by `occupation` → One-hot encoded | Multi-category feature |
| `fnlwgt` | Dropped | Found to be not useful for prediction |
| `education` | One-hot encoded (drop-first) | Categorical feature |
| `education-num` | Standardized + MinMax | Numeric version of education — used instead of string |
| `marital-status` | One-hot encoded | Categorical |
| `occupation` | Replaced '?' → Imputed using `income` & `native-country` → One-hot encoded | Multi-category |
| `relationship` | One-hot encoded | Categorical |
| `race` | One-hot encoded + Used for fairness check + Used to impute `native-country` | Categorical |
| `sex` | Label encoded (Male→0, Female→1) | Binary |
| `capital-gain` | log1p transformed | Very skewed |
| `capital-loss` | log1p transformed | Skewed |
| `hours-per-week` | scaled | Mild outliers |
| `native-country` | Imputed by most common per race → One-hot encoded | Many categories |
| `income` | Label encoded (`<=50K` → 0, `>50K` → 1) | Target variable |
| `capital_net` | Created new feature = `capital-gain` - `capital-loss` | Derived income signal |

---



## Why We Used Encoding

| Encoding Type | Applied To | Why |
|---------------|------------|-----|
| Label Encoding | `income`, `sex` | Only 2 values, so binary 0/1 is fine |
| One-Hot Encoding | `education`, `occupation`, etc. | Multiple unordered categories |
| drop_first=True | All one-hot encoding | Avoid multicollinearity in models |

---



## Why We Used StandardScaler and MinMaxScaler

| Scaler | Columns | Use Case |
|--------|---------|----------|
| `StandardScaler` | `age`, `education-num`, `hours-per-week` | Good for Logistic Regression, SVM |
| `MinMaxScaler` | Same columns | Good for KNN, Neural Nets |
| No scaling | Tree models (RF, XGBoost) | Trees are scale-insensitive |

We created 3 versions:
- `df` (raw cleaned): use for Random Forest / XGBoost
- `df_std`: use for Logistic Regression / SVM
- `df_norm`: use for KNN / Neural Networks

---



###  Education Number and Education are just the same, so, Education of them column could be droped.

In [42]:
if 'education' in df.columns:
    df.drop('education', axis=1, inplace=True)

if 'education' in df_std.columns:
    df_std.drop('education', axis=1, inplace=True)

if 'education' in df_norm.columns:
    df_norm.drop('education', axis=1, inplace=True)

In [43]:
# Check All Current Columns
print('education' in df.columns)
print(df.columns.tolist())  # See what’s in your DataFrame

False
['age', 'education-num', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'income', 'capital_net', 'workclass_local-gov', 'workclass_never-worked', 'workclass_private', 'workclass_self-emp-inc', 'workclass_self-emp-not-inc', 'workclass_state-gov', 'workclass_without-pay', 'education_assoc-voc', 'education_bachelors', 'education_doctorate', 'education_elementary School', 'education_masters', 'education_middle/high School', 'education_preschool', 'education_prof-school', 'marital-status_married-af-spouse', 'marital-status_married-civ-spouse', 'marital-status_married-spouse-absent', 'marital-status_never-married', 'marital-status_separated', 'marital-status_widowed', 'occupation_craft-repair', 'occupation_exec-managerial', 'occupation_farming-fishing', 'occupation_handlers-cleaners', 'occupation_machine-op-inspct', 'occupation_other-service', 'occupation_prof-specialty', 'occupation_sales', 'occupation_transport-moving', 'relationship_not-in-family', 'relationship_other-rela

### Convert True/False to 0/1(bool) for encoded categorical columns.

In [28]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

### Convert True/False to 0/1 in scaled datasets

In [29]:
# Convert True/False to 0/1 in scaled datasets
for df_variant in [df_std, df_norm]:
    bool_cols = df_variant.select_dtypes(include='bool').columns
    df_variant[bool_cols] = df_variant[bool_cols].astype(int)


### To List All Columns with Missing Values after cleaning:

In [30]:
df.isna().sum()[df.isna().sum() > 0]

Series([], dtype: int64)

In [31]:
### Check NaN after cleaning

In [32]:
print(df['sex'].unique())
print(df['income'].unique())

[0 1]
[0 1]


In [33]:
print(df['sex'].unique())
print(df['income'].unique())


[0 1]
[0 1]


In [34]:
# Check object column Types
print(df.dtypes[df.dtypes == 'object'])

Series([], dtype: object)


In [35]:
# check all column data type
print([df.dtypes])

[age                                       int64
education-num                             int64
sex                                       int64
capital-gain                            float64
capital-loss                            float64
hours-per-week                            int64
income                                    int64
capital_net                             float64
workclass_local-gov                       int64
workclass_never-worked                    int64
workclass_private                         int64
workclass_self-emp-inc                    int64
workclass_self-emp-not-inc                int64
workclass_state-gov                       int64
workclass_without-pay                     int64
education_assoc-voc                       int64
education_bachelors                       int64
education_doctorate                       int64
education_elementary School               int64
education_masters                         int64
education_middle/high School           

In [36]:
print(df.select_dtypes(include='bool'))  # should be empty now

Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, ...]

[48813 rows x 0 columns]


In [37]:
df.dtypes.value_counts()

int64      52
float64     3
Name: count, dtype: int64

In [38]:
print(df['income'].unique())

[0 1]


## Saving Final Datasets for Modeling

We'll save all three versions of the dataset for next Assignment


In [39]:
# Save all versions
df.to_csv('df_cleaned.csv', index=False)
df_std.to_csv('df_std.csv', index=False)
df_norm.to_csv('df_norm.csv', index=False)

print("Saved: df_cleaned.csv, df_std.csv, df_norm.csv")


Saved: df_cleaned.csv, df_std.csv, df_norm.csv


In [40]:
df.head()

,age,education-num,sex,capital-gain,capital-loss,hours-per-week,income,capital_net,workclass_local-gov,workclass_never-worked,...,native-country_cuba,native-country_el-salvador,native-country_england,native-country_germany,native-country_india,native-country_mexico,native-country_other,native-country_philippines,native-country_puerto-rico,native-country_united-states
0,39,13,0,7.684784,0.0,40,0,7.684784,0,0,...,0,0,0,0,0,0,0,0,0,1
1,50,13,0,0.000000,0.0,13,0,0.000000,0,0,...,0,0,0,0,0,0,0,0,0,1
2,38,9,0,0.000000,0.0,40,0,0.000000,0,0,...,0,0,0,0,0,0,0,0,0,1
3,53,7,0,0.000000,0.0,40,0,0.000000,0,0,...,0,0,0,0,0,0,0,0,0,1
4,28,13,1,0.000000,0.0,40,0,0.000000,0,0,...,1,0,0,0,0,0,0,0,0,0


In [41]:
# race dummies
print([col for col in df.columns if 'race_' in col])

['race_other', 'race_white']


## Final Column Summary

All transformations are now complete. Each column is either:
- Cleaned the dataset completely
- Applied appropriate encoding
- Handled outliers and skewness
- Engineered features
- Scaled data for various model types
- Saved datasets ready for next assignment

You're now ready to build models in Assignment Model Building!
